In [18]:
#!/usr/bin/env python3
"""
Financial Data Validation & Work Automation Bot
Combined script for financial data management and work percentage tracking
"""

import pandas as pd
import os
import json
from datetime import datetime
import numpy as np
import re

class FinancialAutomationBot:
    def __init__(self, data_file='financial_data.json'):
        self.data_file = data_file
        self.data = self.load_data()
        self.work_log = self.load_work_log()
        
    def load_data(self):
        """Load existing financial data from JSON file"""
        if os.path.exists(self.data_file):
            try:
                with open(self.data_file, 'r') as f:
                    return json.load(f)
            except:
                return []
        return []
    
    def load_work_log(self):
        """Load work log for percentage tracking"""
        if os.path.exists('work_log.json'):
            try:
                with open('work_log.json', 'r') as f:
                    return json.load(f)
            except:
                return {"tasks": [], "completed_hours": 0, "total_hours": 0}
        return {"tasks": [], "completed_hours": 0, "total_hours": 0}
    
    def save_data(self):
        """Save financial data to JSON file"""
        with open(self.data_file, 'w') as f:
            json.dump(self.data, f, indent=4)
    
    def save_work_log(self):
        """Save work log to JSON file"""
        with open('work_log.json', 'w') as f:
            json.dump(self.work_log, f, indent=4)
    
    def add_financial_record(self, record):
        """Add a new financial record"""
        required_fields = ['date', 'amount', 'description']
        for field in required_fields:
            if field not in record or not record[field]:
                return False, f"Missing required field: {field}"
        
        # Validate amount is numeric
        try:
            record['amount'] = float(record['amount'])
        except:
            return False, "Amount must be a number"
        
        # Add timestamp
        record['created_at'] = datetime.now().isoformat()
        
        self.data.append(record)
        self.save_data()
        return True, f"✓ Financial record added: {record['description']}"
    
    def add_task(self, task_name, estimated_hours):
        """Add a task for percentage tracking"""
        task = {
            'id': len(self.work_log['tasks']) + 1,
            'name': task_name,
            'estimated_hours': float(estimated_hours),
            'completed_hours': 0,
            'status': 'pending',
            'created_at': datetime.now().isoformat()
        }
        self.work_log['tasks'].append(task)
        self.work_log['total_hours'] = self.work_log.get('total_hours', 0) + float(estimated_hours)
        self.save_work_log()
        return True, f"✓ Task added: {task_name} ({estimated_hours} hours)"
    
    def update_task_progress(self, task_id, hours_completed):
        """Update task progress and calculate percentage"""
        for task in self.work_log['tasks']:
            if task['id'] == task_id:
                task['completed_hours'] += float(hours_completed)
                self.work_log['completed_hours'] = self.work_log.get('completed_hours', 0) + float(hours_completed)
                
                if task['completed_hours'] >= task['estimated_hours']:
                    task['status'] = 'completed'
                    task['completed_hours'] = task['estimated_hours']
                else:
                    task['status'] = 'in_progress'
                
                self.save_work_log()
                percentage = self.get_completion_percentage()
                return True, f"✓ Progress updated! Overall completion: {percentage:.1f}%"
        return False, "Task not found"
    
    def get_completion_percentage(self):
        """Calculate overall work completion percentage"""
        total_hours = self.work_log.get('total_hours', 0)
        if total_hours == 0:
            return 0
        completed_hours = self.work_log.get('completed_hours', 0)
        return (completed_hours / total_hours) * 100
    
    def get_work_summary(self):
        """Get detailed work summary"""
        if not self.work_log.get('tasks'):
            return None
        
        tasks = self.work_log['tasks']
        completed_tasks = sum(1 for t in tasks if t['status'] == 'completed')
        in_progress_tasks = sum(1 for t in tasks if t['status'] == 'in_progress')
        pending_tasks = sum(1 for t in tasks if t['status'] == 'pending')
        
        return {
            'total_tasks': len(tasks),
            'completed_tasks': completed_tasks,
            'in_progress_tasks': in_progress_tasks,
            'pending_tasks': pending_tasks,
            'total_hours': self.work_log.get('total_hours', 0),
            'completed_hours': self.work_log.get('completed_hours', 0),
            'remaining_hours': self.work_log.get('total_hours', 0) - self.work_log.get('completed_hours', 0),
            'completion_percentage': self.get_completion_percentage()
        }
    
    def validate_financial_csv(self, file_path):
        """Validate financial data from CSV file"""
        try:
            # Read the file
            if file_path.endswith('.csv'):
                df = pd.read_csv(file_path)
            elif file_path.endswith(('.xlsx', '.xls')):
                df = pd.read_excel(file_path)
            else:
                return None, {'error': 'Unsupported file format'}
            
            # Basic validation
            missing_values = df.isnull().sum()
            total_missing = missing_values.sum()
            duplicates = df.duplicated().sum()
            
            # Check for negative values in amount column
            negative_amounts = 0
            if 'amount' in df.columns:
                negative_amounts = (df['amount'] < 0).sum()
            
            # Check for invalid dates
            invalid_dates = 0
            if 'date' in df.columns:
                try:
                    pd.to_datetime(df['date'])
                except:
                    invalid_dates = (pd.to_datetime(df['date'], errors='coerce').isna()).sum()
            
            # Generate report
            report = {
                'file': os.path.basename(file_path),
                'rows': len(df),
                'columns': len(df.columns),
                'missing_values': int(total_missing),
                'duplicates': int(duplicates),
                'negative_amounts': int(negative_amounts),
                'invalid_dates': int(invalid_dates),
                'validation_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                'status': 'PASS' if (total_missing == 0 and duplicates == 0 and negative_amounts == 0) else 'FAIL'
            }
            
            # Clean the data
            df_cleaned = df.copy()
            if duplicates > 0:
                df_cleaned = df_cleaned.drop_duplicates()
            
            # Fill missing values if needed
            for col in df_cleaned.columns:
                if df_cleaned[col].dtype in ['float64', 'int64']:
                    df_cleaned[col] = df_cleaned[col].fillna(0)
                else:
                    df_cleaned[col] = df_cleaned[col].fillna('Unknown')
            
            return df_cleaned, report
            
        except Exception as e:
            return None, {'error': str(e), 'status': 'ERROR'}
    
    def generate_validation_report(self, df, report, output_path):
        """Generate Excel validation report"""
        try:
            os.makedirs(os.path.dirname(output_path), exist_ok=True)
            
            with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
                # Cleaned data sheet
                if df is not None:
                    df.to_excel(writer, sheet_name='Cleaned Data', index=False)
                
                # Summary sheet
                summary_data = []
                for key, value in report.items():
                    if key not in ['validation_date']:
                        summary_data.append({'Metric': key.replace('_', ' ').title(), 'Value': value})
                
                summary_df = pd.DataFrame(summary_data)
                summary_df.to_excel(writer, sheet_name='Validation Summary', index=False)
                
                # Statistics sheet
                if df is not None and 'amount' in df.columns:
                    stats = {
                        'Total Amount': df['amount'].sum(),
                        'Average Amount': df['amount'].mean(),
                        'Max Amount': df['amount'].max(),
                        'Min Amount': df['amount'].min(),
                        'Std Deviation': df['amount'].std()
                    }
                    stats_df = pd.DataFrame([stats])
                    stats_df.to_excel(writer, sheet_name='Financial Statistics', index=False)
            
            return True
        except Exception as e:
            print(f"Error generating report: {e}")
            return False
    
    def get_financial_summary(self):
        """Get summary of financial records"""
        if not self.data:
            return None
        
        df = pd.DataFrame(self.data)
        summary = {
            'total_records': len(self.data),
            'total_amount': df['amount'].sum(),
            'average_amount': df['amount'].mean(),
            'max_amount': df['amount'].max(),
            'min_amount': df['amount'].min()
        }
        
        # Category breakdown
        if 'category' in df.columns:
            category_summary = df.groupby('category')['amount'].agg(['sum', 'count']).to_dict()
            summary['categories'] = category_summary
        
        return summary
    
    def export_financial_data(self, filename='financial_export.csv'):
        """Export financial data to CSV"""
        if self.data:
            df = pd.DataFrame(self.data)
            df.to_csv(filename, index=False)
            return f"✓ Exported {len(self.data)} records to {filename}"
        return "✗ No data to export"
    
    def search_records(self, keyword):
        """Search financial records by keyword"""
        results = []
        for record in self.data:
            if keyword.lower() in str(record.get('description', '')).lower():
                results.append(record)
        return results

def print_header(title):
    """Print formatted header"""
    print("\n" + "="*60)
    print(f" {title}")
    print("="*60)

def print_progress_bar(percentage, width=40):
    """Print a progress bar"""
    filled = int(width * percentage / 100)
    bar = '█' * filled + '░' * (width - filled)
    print(f"   [{bar}] {percentage:.1f}%")

def main():
    bot = FinancialAutomationBot()
    
    while True:
        print_header("💰 FINANCIAL DATA VALIDATION & WORK AUTOMATION BOT")
        
        # Show work completion percentage
        percentage = bot.get_completion_percentage()
        print(f"\n📊 OVERALL WORK COMPLETION: {percentage:.1f}%")
        print_progress_bar(percentage)
        
        print(f"\n📋 Financial Records: {len(bot.data)} | Tasks: {len(bot.work_log.get('tasks', []))}")
        
        print("\n📌 MAIN MENU:")
        print("   ┌─────────────────────────────────────────────────┐")
        print("   │ 1.  ➕ Add Financial Record                      │")
        print("   │ 2.  📋 View All Financial Records                │")
        print("   │ 3.  🔍 Search Financial Records                  │")
        print("   │ 4.  📊 Validate CSV/Excel File                   │")
        print("   │ 5.  📈 View Financial Summary                    │")
        print("   │ 6.  💼 Task Management (Work Tracking)           │")
        print("   │ 7.  📉 Work Progress Report                      │")
        print("   │ 8.  📤 Export Financial Data to CSV              │")
        print("   │ 9.  🚪 Exit                                      │")
        print("   └─────────────────────────────────────────────────┘")
        
        choice = input("\n👉 Enter your choice (1-9): ").strip()
        
        # Option 1: Add Financial Record
        if choice == '1':
            print_header("➕ ADD FINANCIAL RECORD")
            print("(Press Enter to skip optional fields)\n")
            
            date = input("Date (YYYY-MM-DD): ").strip()
            amount = input("Amount ($): ").strip()
            description = input("Description: ").strip()
            category = input("Category (Income/Expense/Other): ").strip() or "Other"
            
            record = {
                'date': date,
                'amount': amount,
                'description': description,
                'category': category
            }
            
            success, message = bot.add_financial_record(record)
            print(f"\n{message}")
        
        # Option 2: View All Records
        elif choice == '2':
            print_header("📋 FINANCIAL RECORDS")
            if bot.data:
                for i, record in enumerate(bot.data[-20:], 1):
                    amount_val = record.get('amount', 0)
                    print(f"\n{i}. 📅 {record.get('date', 'N/A')}")
                    print(f"   💰 Amount: ${amount_val:,.2f}")
                    print(f"   📝 Description: {record.get('description', 'N/A')}")
                    print(f"   🏷️  Category: {record.get('category', 'N/A')}")
            else:
                print("\n📭 No financial records found")
        
        # Option 3: Search Records
        elif choice == '3':
            print_header("🔍 SEARCH FINANCIAL RECORDS")
            keyword = input("Enter search keyword: ").strip()
            results = bot.search_records(keyword)
            
            if results:
                print(f"\n✅ Found {len(results)} result(s):")
                for result in results:
                    amount_val = result.get('amount', 0)
                    print(f"   • {result.get('date')}: ${amount_val:,.2f} - {result.get('description')}")
            else:
                print("\n❌ No matching records found")
        
        # Option 4: Validate CSV/Excel File
        elif choice == '4':
            print_header("📊 VALIDATE CSV/EXCEL FILE")
            print("Supported formats: .csv, .xlsx, .xls")
            filename = input("Enter filename (in 'input' folder): ").strip()
            file_path = f"./input/{filename}"
            
            if os.path.exists(file_path):
                print(f"\n⏳ Processing {filename}...")
                df, report = bot.validate_financial_csv(file_path)
                
                if df is not None and report.get('status') != 'ERROR':
                    print(f"\n✅ VALIDATION REPORT:")
                    print(f"   ┌─────────────────────────────────────────┐")
                    print(f"   │ File: {report['file']:<35}│")
                    print(f"   │ Rows: {report['rows']:<38}│")
                    print(f"   │ Columns: {report['columns']:<35}│")
                    print(f"   │ Missing Values: {report['missing_values']:<28}│")
                    print(f"   │ Duplicates: {report['duplicates']:<32}│")
                    print(f"   │ Negative Amounts: {report['negative_amounts']:<26}│")
                    print(f"   │ Invalid Dates: {report['invalid_dates']:<29}│")
                    print(f"   │ Status: {report['status']:<36}│")
                    print(f"   └─────────────────────────────────────────┘")
                    
                    # Save validated file
                    os.makedirs('./output', exist_ok=True)
                    base_name = filename.replace('.csv', '').replace('.xlsx', '').replace('.xls', '')
                    output_path = f"./output/validated_{base_name}.xlsx"
                    bot.generate_validation_report(df, report, output_path)
                    print(f"\n💾 Validated report saved to: {output_path}")
                else:
                    error_msg = report.get('error', 'Unknown error')
                    print(f"\n❌ Error: {error_msg}")
            else:
                print(f"\n❌ File not found: {file_path}")
                print("   Make sure the file is in the './input' folder")
        
        # Option 5: Financial Summary
        elif choice == '5':
            print_header("📈 FINANCIAL SUMMARY")
            summary = bot.get_financial_summary()
            
            if summary:
                total_amount = summary['total_amount']
                avg_amount = summary['average_amount']
                max_amount = summary['max_amount']
                min_amount = summary['min_amount']
                
                print(f"\n   📊 OVERALL STATISTICS:")
                print(f"   ┌─────────────────────────────────────────┐")
                print(f"   │ Total Records: {summary['total_records']:<30}│")
                print(f"   │ Total Amount: ${total_amount:>15,.2f} │")
                print(f"   │ Average Amount: ${avg_amount:>14,.2f} │")
                print(f"   │ Maximum Amount: ${max_amount:>14,.2f} │")
                print(f"   │ Minimum Amount: ${min_amount:>14,.2f} │")
                print(f"   └─────────────────────────────────────────┘")
                
                if 'categories' in summary:
                    print(f"\n   📂 CATEGORY BREAKDOWN:")
                    for category, data in summary['categories']['sum'].items():
                        count = summary['categories']['count'][category]
                        print(f"   • {category}: ${data:,.2f} ({count} records)")
            else:
                print("\n📭 No financial data available")
        
        # Option 6: Task Management
        elif choice == '6':
            print_header("💼 TASK MANAGEMENT (WORK TRACKING)")
            print("\n   Task Management Options:")
            print("   ┌─────────────────────────────────────────┐")
            print("   │ A. Add New Task                          │")
            print("   │ B. Update Task Progress                  │")
            print("   │ C. View All Tasks                        │")
            print("   │ D. Back to Main Menu                     │")
            print("   └─────────────────────────────────────────┘")
            
            task_choice = input("\n👉 Choice (A/B/C/D): ").strip().upper()
            
            if task_choice == 'A':
                task_name = input("Task name: ").strip()
                hours = input("Estimated hours: ").strip()
                success, message = bot.add_task(task_name, hours)
                print(f"\n{message}")
            
            elif task_choice == 'B':
                tasks = bot.work_log.get('tasks', [])
                if not tasks:
                    print("\n📭 No tasks available. Add a task first.")
                else:
                    print("\n📋 Available Tasks:")
                    for task in tasks:
                        status_icon = "✅" if task['status'] == 'completed' else "🔄" if task['status'] == 'in_progress' else "⏳"
                        print(f"   {status_icon} Task #{task['id']}: {task['name']} - {task['completed_hours']:.1f}/{task['estimated_hours']:.1f} hours")
                    
                    try:
                        task_id = int(input("\nTask ID to update: "))
                        hours = float(input("Hours completed: "))
                        success, message = bot.update_task_progress(task_id, hours)
                        print(f"\n{message}")
                    except ValueError:
                        print("\n❌ Invalid input")
            
            elif task_choice == 'C':
                tasks = bot.work_log.get('tasks', [])
                if not tasks:
                    print("\n📭 No tasks found")
                else:
                    print_header("📋 ALL TASKS")
                    for task in tasks:
                        status_icon = "✅" if task['status'] == 'completed' else "🔄" if task['status'] == 'in_progress' else "⏳"
                        progress = (task['completed_hours'] / task['estimated_hours']) * 100 if task['estimated_hours'] > 0 else 0
                        print(f"\n{status_icon} Task #{task['id']}: {task['name']}")
                        print(f"   Hours: {task['completed_hours']:.1f}/{task['estimated_hours']:.1f} ({progress:.1f}%)")
                        print(f"   Status: {task['status'].upper()}")
                        print(f"   Created: {task['created_at'][:10]}")
        
        # Option 7: Work Progress Report
        elif choice == '7':
            print_header("📉 WORK PROGRESS REPORT")
            summary = bot.get_work_summary()
            
            if summary:
                percentage = summary['completion_percentage']
                print(f"\n   📊 COMPLETION PERCENTAGE: {percentage:.1f}%")
                print_progress_bar(percentage, 50)
                
                print(f"\n   📋 TASK SUMMARY:")
                print(f"   ┌─────────────────────────────────────────┐")
                print(f"   │ Total Tasks: {summary['total_tasks']:<32}│")
                print(f"   │ Completed Tasks: {summary['completed_tasks']:<29}│")
                print(f"   │ In Progress: {summary['in_progress_tasks']:<32}│")
                print(f"   │ Pending Tasks: {summary['pending_tasks']:<31}│")
                print(f"   └─────────────────────────────────────────┘")
                
                print(f"\n   ⏰ HOURS SUMMARY:")
                print(f"   ┌─────────────────────────────────────────┐")
                print(f"   │ Total Hours: {summary['total_hours']:<30.1f}│")
                print(f"   │ Completed Hours: {summary['completed_hours']:<27.1f}│")
                print(f"   │ Remaining Hours: {summary['remaining_hours']:<28.1f}│")
                print(f"   └─────────────────────────────────────────┘")
            else:
                print("\n📭 No work data available. Add tasks to track progress.")
        
        # Option 8: Export Financial Data
        elif choice == '8':
            print_header("📤 EXPORT FINANCIAL DATA")
            filename = input("Filename (default: financial_export.csv): ").strip() or "financial_export.csv"
            if not filename.endswith('.csv'):
                filename += '.csv'
            result = bot.export_financial_data(filename)
            print(f"\n{result}")
        
        # Option 9: Exit
        elif choice == '9':
            final_percentage = bot.get_completion_percentage()
            print_header("👋 GOODBYE!")
            print(f"\n📊 Final Work Completion: {final_percentage:.1f}%")
            print_progress_bar(final_percentage)
            print("\n✅ Thanks for using Financial Data Validation & Work Automation Bot!")
            print("💾 All data has been saved automatically.")
            break
        
        else:
            print("\n❌ Invalid choice. Please enter a number between 1 and 9.")
        
        input("\n⏎ Press Enter to continue...")

if __name__ == "__main__":
    # Create necessary folders
    os.makedirs('./input', exist_ok=True)
    os.makedirs('./output', exist_ok=True)
    
    print("\n" + "="*60)
    print(" FINANCIAL DATA VALIDATION & WORK AUTOMATION BOT")
    print("="*60)
    print("\n📁 Folder Structure Created:")
    print("   📂 ./input/  - Place your CSV/Excel files here for validation")
    print("   📂 ./output/ - Validated reports will be saved here")
    print("\n💾 Data Files:")
    print("   📄 financial_data.json - Stores your financial records")
    print("   📄 work_log.json - Tracks your work progress")
    print("\n" + "="*60)
    
    main()


 💰 FINANCIAL DATA VALIDATION BOT

📊 WORK COMPLETION: 0.0%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

📋 Records in Database: 0

📌 MAIN MENU:
  1. ➕ Add Financial Record
  2. 📋 View All Records
  3. 🔍 Search Records
  4. 📊 Validate CSV File
  5. 📈 View Financial Summary
  6. 💼 Task Management
  7. 📊 Export to CSV
  8. 📉 Work Progress Report
  9. 🚪 Exit


KeyboardInterrupt: Interrupted by user

In [ ]:
#!/usr/bin/env python3
"""
Financial Data Validation & Work Automation Bot
Combined script for financial data management and work percentage tracking
"""

import pandas as pd
import os
import json
from datetime import datetime
import numpy as np
import re

class FinancialAutomationBot:
    def __init__(self, data_file='financial_data.json'):
        self.data_file = data_file
        self.data = self.load_data()
        self.work_log = self.load_work_log()
        
    def load_data(self):
        """Load existing financial data from JSON file"""
        if os.path.exists(self.data_file):
            try:
                with open(self.data_file, 'r') as f:
                    return json.load(f)
            except:
                return []
        return []
    
    def load_work_log(self):
        """Load work log for percentage tracking"""
        if os.path.exists('work_log.json'):
            try:
                with open('work_log.json', 'r') as f:
                    return json.load(f)
            except:
                return {"tasks": [], "completed_hours": 0, "total_hours": 0}
        return {"tasks": [], "completed_hours": 0, "total_hours": 0}
    
    def save_data(self):
        """Save financial data to JSON file"""
        with open(self.data_file, 'w') as f:
            json.dump(self.data, f, indent=4)
    
    def save_work_log(self):
        """Save work log to JSON file"""
        with open('work_log.json', 'w') as f:
            json.dump(self.work_log, f, indent=4)
    
    def add_financial_record(self, record):
        """Add a new financial record"""
        required_fields = ['date', 'amount', 'description']
        for field in required_fields:
            if field not in record or not record[field]:
                return False, f"Missing required field: {field}"
        
        # Validate amount is numeric
        try:
            record['amount'] = float(record['amount'])
        except:
            return False, "Amount must be a number"
        
        # Add timestamp
        record['created_at'] = datetime.now().isoformat()
        
        self.data.append(record)
        self.save_data()
        return True, f"✓ Financial record added: {record['description']}"
    
    def add_task(self, task_name, estimated_hours):
        """Add a task for percentage tracking"""
        task = {
            'id': len(self.work_log['tasks']) + 1,
            'name': task_name,
            'estimated_hours': float(estimated_hours),
            'completed_hours': 0,
            'status': 'pending',
            'created_at': datetime.now().isoformat()
        }
        self.work_log['tasks'].append(task)
        self.work_log['total_hours'] = self.work_log.get('total_hours', 0) + float(estimated_hours)
        self.save_work_log()
        return True, f"✓ Task added: {task_name} ({estimated_hours} hours)"
    
    def update_task_progress(self, task_id, hours_completed):
        """Update task progress and calculate percentage"""
        for task in self.work_log['tasks']:
            if task['id'] == task_id:
                task['completed_hours'] += float(hours_completed)
                self.work_log['completed_hours'] = self.work_log.get('completed_hours', 0) + float(hours_completed)
                
                if task['completed_hours'] >= task['estimated_hours']:
                    task['status'] = 'completed'
                    task['completed_hours'] = task['estimated_hours']
                else:
                    task['status'] = 'in_progress'
                
                self.save_work_log()
                percentage = self.get_completion_percentage()
                return True, f"✓ Progress updated! Overall completion: {percentage:.1f}%"
        return False, "Task not found"
    
    def get_completion_percentage(self):
        """Calculate overall work completion percentage"""
        total_hours = self.work_log.get('total_hours', 0)
        if total_hours == 0:
            return 0
        completed_hours = self.work_log.get('completed_hours', 0)
        return (completed_hours / total_hours) * 100
    
    def get_work_summary(self):
        """Get detailed work summary"""
        if not self.work_log.get('tasks'):
            return None
        
        tasks = self.work_log['tasks']
        completed_tasks = sum(1 for t in tasks if t['status'] == 'completed')
        in_progress_tasks = sum(1 for t in tasks if t['status'] == 'in_progress')
        pending_tasks = sum(1 for t in tasks if t['status'] == 'pending')
        
        return {
            'total_tasks': len(tasks),
            'completed_tasks': completed_tasks,
            'in_progress_tasks': in_progress_tasks,
            'pending_tasks': pending_tasks,
            'total_hours': self.work_log.get('total_hours', 0),
            'completed_hours': self.work_log.get('completed_hours', 0),
            'remaining_hours': self.work_log.get('total_hours', 0) - self.work_log.get('completed_hours', 0),
            'completion_percentage': self.get_completion_percentage()
        }
    
    def validate_financial_csv(self, file_path):
        """Validate financial data from CSV file"""
        try:
            # Read the file
            if file_path.endswith('.csv'):
                df = pd.read_csv(file_path)
            elif file_path.endswith(('.xlsx', '.xls')):
                df = pd.read_excel(file_path)
            else:
                return None, {'error': 'Unsupported file format'}
            
            # Basic validation
            missing_values = df.isnull().sum()
            total_missing = missing_values.sum()
            duplicates = df.duplicated().sum()
            
            # Check for negative values in amount column
            negative_amounts = 0
            if 'amount' in df.columns:
                negative_amounts = (df['amount'] < 0).sum()
            
            # Check for invalid dates
            invalid_dates = 0
            if 'date' in df.columns:
                try:
                    pd.to_datetime(df['date'])
                except:
                    invalid_dates = (pd.to_datetime(df['date'], errors='coerce').isna()).sum()
            
            # Generate report
            report = {
                'file': os.path.basename(file_path),
                'rows': len(df),
                'columns': len(df.columns),
                'missing_values': int(total_missing),
                'duplicates': int(duplicates),
                'negative_amounts': int(negative_amounts),
                'invalid_dates': int(invalid_dates),
                'validation_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                'status': 'PASS' if (total_missing == 0 and duplicates == 0 and negative_amounts == 0) else 'FAIL'
            }
            
            # Clean the data
            df_cleaned = df.copy()
            if duplicates > 0:
                df_cleaned = df_cleaned.drop_duplicates()
            
            # Fill missing values if needed
            for col in df_cleaned.columns:
                if df_cleaned[col].dtype in ['float64', 'int64']:
                    df_cleaned[col] = df_cleaned[col].fillna(0)
                else:
                    df_cleaned[col] = df_cleaned[col].fillna('Unknown')
            
            return df_cleaned, report
            
        except Exception as e:
            return None, {'error': str(e), 'status': 'ERROR'}
    
    def generate_validation_report(self, df, report, output_path):
        """Generate Excel validation report"""
        try:
            os.makedirs(os.path.dirname(output_path), exist_ok=True)
            
            with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
                # Cleaned data sheet
                if df is not None:
                    df.to_excel(writer, sheet_name='Cleaned Data', index=False)
                
                # Summary sheet
                summary_data = []
                for key, value in report.items():
                    if key not in ['validation_date']:
                        summary_data.append({'Metric': key.replace('_', ' ').title(), 'Value': value})
                
                summary_df = pd.DataFrame(summary_data)
                summary_df.to_excel(writer, sheet_name='Validation Summary', index=False)
                
                # Statistics sheet
                if df is not None and 'amount' in df.columns:
                    stats = {
                        'Total Amount': df['amount'].sum(),
                        'Average Amount': df['amount'].mean(),
                        'Max Amount': df['amount'].max(),
                        'Min Amount': df['amount'].min(),
                        'Std Deviation': df['amount'].std()
                    }
                    stats_df = pd.DataFrame([stats])
                    stats_df.to_excel(writer, sheet_name='Financial Statistics', index=False)
            
            return True
        except Exception as e:
            print(f"Error generating report: {e}")
            return False
    
    def get_financial_summary(self):
        """Get summary of financial records"""
        if not self.data:
            return None
        
        df = pd.DataFrame(self.data)
        summary = {
            'total_records': len(self.data),
            'total_amount': df['amount'].sum(),
            'average_amount': df['amount'].mean(),
            'max_amount': df['amount'].max(),
            'min_amount': df['amount'].min()
        }
        
        # Category breakdown
        if 'category' in df.columns:
            category_summary = df.groupby('category')['amount'].agg(['sum', 'count']).to_dict()
            summary['categories'] = category_summary
        
        return summary
    
    def export_financial_data(self, filename='financial_export.csv'):
        """Export financial data to CSV"""
        if self.data:
            df = pd.DataFrame(self.data)
            df.to_csv(filename, index=False)
            return f"✓ Exported {len(self.data)} records to {filename}"
        return "✗ No data to export"
    
    def search_records(self, keyword):
        """Search financial records by keyword"""
        results = []
        for record in self.data:
            if keyword.lower() in str(record.get('description', '')).lower():
                results.append(record)
        return results

def print_header(title):
    """Print formatted header"""
    print("\n" + "="*60)
    print(f" {title}")
    print("="*60)

def print_progress_bar(percentage, width=40):
    """Print a progress bar"""
    filled = int(width * percentage / 100)
    bar = '█' * filled + '░' * (width - filled)
    print(f"   [{bar}] {percentage:.1f}%")

def main():
    bot = FinancialAutomationBot()
    
    while True:
        print_header("💰 FINANCIAL DATA VALIDATION & WORK AUTOMATION BOT")
        
        # Show work completion percentage
        percentage = bot.get_completion_percentage()
        print(f"\n📊 OVERALL WORK COMPLETION: {percentage:.1f}%")
        print_progress_bar(percentage)
        
        print(f"\n📋 Financial Records: {len(bot.data)} | Tasks: {len(bot.work_log.get('tasks', []))}")
        
        print("\n📌 MAIN MENU:")
        print("   ┌─────────────────────────────────────────────────┐")
        print("   │ 1.  ➕ Add Financial Record                      │")
        print("   │ 2.  📋 View All Financial Records                │")
        print("   │ 3.  🔍 Search Financial Records                  │")
        print("   │ 4.  📊 Validate CSV/Excel File                   │")
        print("   │ 5.  📈 View Financial Summary                    │")
        print("   │ 6.  💼 Task Management (Work Tracking)           │")
        print("   │ 7.  📉 Work Progress Report                      │")
        print("   │ 8.  📤 Export Financial Data to CSV              │")
        print("   │ 9.  🚪 Exit                                      │")
        print("   └─────────────────────────────────────────────────┘")
        
        choice = input("\n👉 Enter your choice (1-9): ").strip()
        
        # Option 1: Add Financial Record
        if choice == '1':
            print_header("➕ ADD FINANCIAL RECORD")
            print("(Press Enter to skip optional fields)\n")
            
            date = input("Date (YYYY-MM-DD): ").strip()
            amount = input("Amount ($): ").strip()
            description = input("Description: ").strip()
            category = input("Category (Income/Expense/Other): ").strip() or "Other"
            
            record = {
                'date': date,
                'amount': amount,
                'description': description,
                'category': category
            }
            
            success, message = bot.add_financial_record(record)
            print(f"\n{message}")
        
        # Option 2: View All Records
        elif choice == '2':
            print_header("📋 FINANCIAL RECORDS")
            if bot.data:
                for i, record in enumerate(bot.data[-20:], 1):
                    amount_val = record.get('amount', 0)
                    print(f"\n{i}. 📅 {record.get('date', 'N/A')}")
                    print(f"   💰 Amount: ${amount_val:,.2f}")
                    print(f"   📝 Description: {record.get('description', 'N/A')}")
                    print(f"   🏷️  Category: {record.get('category', 'N/A')}")
            else:
                print("\n📭 No financial records found")
        
        # Option 3: Search Records
        elif choice == '3':
            print_header("🔍 SEARCH FINANCIAL RECORDS")
            keyword = input("Enter search keyword: ").strip()
            results = bot.search_records(keyword)
            
            if results:
                print(f"\n✅ Found {len(results)} result(s):")
                for result in results:
                    amount_val = result.get('amount', 0)
                    print(f"   • {result.get('date')}: ${amount_val:,.2f} - {result.get('description')}")
            else:
                print("\n❌ No matching records found")
        
        # Option 4: Validate CSV/Excel File
        elif choice == '4':
            print_header("📊 VALIDATE CSV/EXCEL FILE")
            print("Supported formats: .csv, .xlsx, .xls")
            filename = input("Enter filename (in 'input' folder): ").strip()
            file_path = f"./input/{filename}"
            
            if os.path.exists(file_path):
                print(f"\n⏳ Processing {filename}...")
                df, report = bot.validate_financial_csv(file_path)
                
                if df is not None and report.get('status') != 'ERROR':
                    print(f"\n✅ VALIDATION REPORT:")
                    print(f"   ┌─────────────────────────────────────────┐")
                    print(f"   │ File: {report['file']:<35}│")
                    print(f"   │ Rows: {report['rows']:<38}│")
                    print(f"   │ Columns: {report['columns']:<35}│")
                    print(f"   │ Missing Values: {report['missing_values']:<28}│")
                    print(f"   │ Duplicates: {report['duplicates']:<32}│")
                    print(f"   │ Negative Amounts: {report['negative_amounts']:<26}│")
                    print(f"   │ Invalid Dates: {report['invalid_dates']:<29}│")
                    print(f"   │ Status: {report['status']:<36}│")
                    print(f"   └─────────────────────────────────────────┘")
                    
                    # Save validated file
                    os.makedirs('./output', exist_ok=True)
                    base_name = filename.replace('.csv', '').replace('.xlsx', '').replace('.xls', '')
                    output_path = f"./output/validated_{base_name}.xlsx"
                    bot.generate_validation_report(df, report, output_path)
                    print(f"\n💾 Validated report saved to: {output_path}")
                else:
                    error_msg = report.get('error', 'Unknown error')
                    print(f"\n❌ Error: {error_msg}")
            else:
                print(f"\n❌ File not found: {file_path}")
                print("   Make sure the file is in the './input' folder")
        
        # Option 5: Financial Summary
        elif choice == '5':
            print_header("📈 FINANCIAL SUMMARY")
            summary = bot.get_financial_summary()
            
            if summary:
                total_amount = summary['total_amount']
                avg_amount = summary['average_amount']
                max_amount = summary['max_amount']
                min_amount = summary['min_amount']
                
                print(f"\n   📊 OVERALL STATISTICS:")
                print(f"   ┌─────────────────────────────────────────┐")
                print(f"   │ Total Records: {summary['total_records']:<30}│")
                print(f"   │ Total Amount: ${total_amount:>15,.2f} │")
                print(f"   │ Average Amount: ${avg_amount:>14,.2f} │")
                print(f"   │ Maximum Amount: ${max_amount:>14,.2f} │")
                print(f"   │ Minimum Amount: ${min_amount:>14,.2f} │")
                print(f"   └─────────────────────────────────────────┘")
                
                if 'categories' in summary:
                    print(f"\n   📂 CATEGORY BREAKDOWN:")
                    for category, data in summary['categories']['sum'].items():
                        count = summary['categories']['count'][category]
                        print(f"   • {category}: ${data:,.2f} ({count} records)")
            else:
                print("\n📭 No financial data available")
        
        # Option 6: Task Management
        elif choice == '6':
            print_header("💼 TASK MANAGEMENT (WORK TRACKING)")
            print("\n   Task Management Options:")
            print("   ┌─────────────────────────────────────────┐")
            print("   │ A. Add New Task                          │")
            print("   │ B. Update Task Progress                  │")
            print("   │ C. View All Tasks                        │")
            print("   │ D. Back to Main Menu                     │")
            print("   └─────────────────────────────────────────┘")
            
            task_choice = input("\n👉 Choice (A/B/C/D): ").strip().upper()
            
            if task_choice == 'A':
                task_name = input("Task name: ").strip()
                hours = input("Estimated hours: ").strip()
                success, message = bot.add_task(task_name, hours)
                print(f"\n{message}")
            
            elif task_choice == 'B':
                tasks = bot.work_log.get('tasks', [])
                if not tasks:
                    print("\n📭 No tasks available. Add a task first.")
                else:
                    print("\n📋 Available Tasks:")
                    for task in tasks:
                        status_icon = "✅" if task['status'] == 'completed' else "🔄" if task['status'] == 'in_progress' else "⏳"
                        print(f"   {status_icon} Task #{task['id']}: {task['name']} - {task['completed_hours']:.1f}/{task['estimated_hours']:.1f} hours")
                    
                    try:
                        task_id = int(input("\nTask ID to update: "))
                        hours = float(input("Hours completed: "))
                        success, message = bot.update_task_progress(task_id, hours)
                        print(f"\n{message}")
                    except ValueError:
                        print("\n❌ Invalid input")
            
            elif task_choice == 'C':
                tasks = bot.work_log.get('tasks', [])
                if not tasks:
                    print("\n📭 No tasks found")
                else:
                    print_header("📋 ALL TASKS")
                    for task in tasks:
                        status_icon = "✅" if task['status'] == 'completed' else "🔄" if task['status'] == 'in_progress' else "⏳"
                        progress = (task['completed_hours'] / task['estimated_hours']) * 100 if task['estimated_hours'] > 0 else 0
                        print(f"\n{status_icon} Task #{task['id']}: {task['name']}")
                        print(f"   Hours: {task['completed_hours']:.1f}/{task['estimated_hours']:.1f} ({progress:.1f}%)")
                        print(f"   Status: {task['status'].upper()}")
                        print(f"   Created: {task['created_at'][:10]}")
        
        # Option 7: Work Progress Report
        elif choice == '7':
            print_header("📉 WORK PROGRESS REPORT")
            summary = bot.get_work_summary()
            
            if summary:
                percentage = summary['completion_percentage']
                print(f"\n   📊 COMPLETION PERCENTAGE: {percentage:.1f}%")
                print_progress_bar(percentage, 50)
                
                print(f"\n   📋 TASK SUMMARY:")
                print(f"   ┌─────────────────────────────────────────┐")
                print(f"   │ Total Tasks: {summary['total_tasks']:<32}│")
                print(f"   │ Completed Tasks: {summary['completed_tasks']:<29}│")
                print(f"   │ In Progress: {summary['in_progress_tasks']:<32}│")
                print(f"   │ Pending Tasks: {summary['pending_tasks']:<31}│")
                print(f"   └─────────────────────────────────────────┘")
                
                print(f"\n   ⏰ HOURS SUMMARY:")
                print(f"   ┌─────────────────────────────────────────┐")
                print(f"   │ Total Hours: {summary['total_hours']:<30.1f}│")
                print(f"   │ Completed Hours: {summary['completed_hours']:<27.1f}│")
                print(f"   │ Remaining Hours: {summary['remaining_hours']:<28.1f}│")
                print(f"   └─────────────────────────────────────────┘")
            else:
                print("\n📭 No work data available. Add tasks to track progress.")
        
        # Option 8: Export Financial Data
        elif choice == '8':
            print_header("📤 EXPORT FINANCIAL DATA")
            filename = input("Filename (default: financial_export.csv): ").strip() or "financial_export.csv"
            if not filename.endswith('.csv'):
                filename += '.csv'
            result = bot.export_financial_data(filename)
            print(f"\n{result}")
        
        # Option 9: Exit
        elif choice == '9':
            final_percentage = bot.get_completion_percentage()
            print_header("👋 GOODBYE!")
            print(f"\n📊 Final Work Completion: {final_percentage:.1f}%")
            print_progress_bar(final_percentage)
            print("\n✅ Thanks for using Financial Data Validation & Work Automation Bot!")
            print("💾 All data has been saved automatically.")
            break
        
        else:
            print("\n❌ Invalid choice. Please enter a number between 1 and 9.")
        
        input("\n⏎ Press Enter to continue...")

if __name__ == "__main__":
    # Create necessary folders
    os.makedirs('./input', exist_ok=True)
    os.makedirs('./output', exist_ok=True)
    
    print("\n" + "="*60)
    print(" FINANCIAL DATA VALIDATION & WORK AUTOMATION BOT")
    print("="*60)
    print("\n📁 Folder Structure Created:")
    print("   📂 ./input/  - Place your CSV/Excel files here for validation")
    print("   📂 ./output/ - Validated reports will be saved here")
    print("\n💾 Data Files:")
    print("   📄 financial_data.json - Stores your financial records")
    print("   📄 work_log.json - Tracks your work progress")
    print("\n" + "="*60)
    
    main()


 FINANCIAL DATA VALIDATION & WORK AUTOMATION BOT

📁 Folder Structure Created:
   📂 ./input/  - Place your CSV/Excel files here for validation
   📂 ./output/ - Validated reports will be saved here

💾 Data Files:
   📄 financial_data.json - Stores your financial records
   📄 work_log.json - Tracks your work progress


 💰 FINANCIAL DATA VALIDATION & WORK AUTOMATION BOT

📊 OVERALL WORK COMPLETION: 0.0%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 0.0%

📋 Financial Records: 0 | Tasks: 0

📌 MAIN MENU:
   ┌─────────────────────────────────────────────────┐
   │ 1.  ➕ Add Financial Record                      │
   │ 2.  📋 View All Financial Records                │
   │ 3.  🔍 Search Financial Records                  │
   │ 4.  📊 Validate CSV/Excel File                   │
   │ 5.  📈 View Financial Summary                    │
   │ 6.  💼 Task Management (Work Tracking)           │
   │ 7.  📉 Work Progress Report                      │
   │ 8.  📤 Export Financial Data to CSV              │
   


👉 Enter your choice (1-9):  1



 ➕ ADD FINANCIAL RECORD
(Press Enter to skip optional fields)



Date (YYYY-MM-DD):  2025-11-10
Amount ($):  755
Description:  Tuckshop
Category (Income/Expense/Other):  Income



✓ Financial record added: Tuckshop



⏎ Press Enter to continue... 



 💰 FINANCIAL DATA VALIDATION & WORK AUTOMATION BOT

📊 OVERALL WORK COMPLETION: 0.0%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 0.0%

📋 Financial Records: 1 | Tasks: 0

📌 MAIN MENU:
   ┌─────────────────────────────────────────────────┐
   │ 1.  ➕ Add Financial Record                      │
   │ 2.  📋 View All Financial Records                │
   │ 3.  🔍 Search Financial Records                  │
   │ 4.  📊 Validate CSV/Excel File                   │
   │ 5.  📈 View Financial Summary                    │
   │ 6.  💼 Task Management (Work Tracking)           │
   │ 7.  📉 Work Progress Report                      │
   │ 8.  📤 Export Financial Data to CSV              │
   │ 9.  🚪 Exit                                      │
   └─────────────────────────────────────────────────┘



👉 Enter your choice (1-9):  5



 📈 FINANCIAL SUMMARY

   📊 OVERALL STATISTICS:
   ┌─────────────────────────────────────────┐
   │ Total Records: 1                             │
   │ Total Amount: $         755.00 │
   │ Average Amount: $        755.00 │
   │ Maximum Amount: $        755.00 │
   │ Minimum Amount: $        755.00 │
   └─────────────────────────────────────────┘

   📂 CATEGORY BREAKDOWN:
   • Income: $755.00 (1 records)
